In [5]:
import pandas as pd
import numpy as np

# STEP 1: Load data
df = pd.read_csv("../data/final_unique_dataset.csv")

In [6]:
# STEP 2: Handle missing values
if "Turbulence" in df.columns:
    df["Turbulence"].fillna(df["Turbulence"].mode()[0], inplace=True)

C:\Users\pg293\AppData\Local\Temp\ipykernel_8712\823942787.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Turbulence"].fillna(df["Turbulence"].mode()[0], inplace=True)


In [7]:
# STEP 3: Convert Turbulence safely
if "Turbulence" in df.columns and df["Turbulence"].dtype == "object":
    df["Turbulence"] = df["Turbulence"].map({
        "None": 0,
        "Low": 1,
        "Light": 1,
        "Moderate": 2,
        "High": 3,
        "Severe": 3
    }).fillna(1)

In [8]:
# STEP 4: Create STRONG Accident logic (robust)
df["Accident"] = (
    ((df.get("Visibility_Km", 10) < 3) & (df.get("Wind_Speed_Kmph", 0) > 50)) |
    ((df.get("Turbulence", 0) >= 2) & (df.get("Pilot_Experience_Hrs", 10000) < 5000))
).astype(int)



In [9]:
# STEP 5: Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])


In [16]:
# STEP 6: Drop useless columns safely
drop_cols = ["Accident", "Flight_ID", "Date", "Time_UTC", "Total_Onboard"]
existing_drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop([
    "Accident",
    "Flight_ID",
    "Date",
    "Time_UTC",
    "Total_Onboard",
    "Visibility_Km",
    "Wind_Speed_Kmph",
    "Turbulence",
    "Pilot_Experience_Hrs"
], axis=1, errors="ignore")

y = df["Accident"]

In [17]:
# STEP 7: Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [24]:
# STEP 8: Train model (optimized)
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=12,
                       min_samples_split=5, n_estimators=300, random_state=42)

In [25]:
# STEP 9: Accuracy
accuracy = model.score(X_test, y_test)
print("🔥 Accuracy:", accuracy)


🔥 Accuracy: 0.7358181818181818


In [26]:
import pickle

# Save model
pickle.dump(model, open("../model/model.pkl", "wb"))

# 🔥 Save scaler (VERY IMPORTANT)
pickle.dump(scaler, open("../model/scaler.pkl", "wb"))

print("✅ Model and scaler saved successfully!")

✅ Model and scaler saved successfully!


In [27]:
print(df["Accident"].value_counts())

Accident
0    42335
1    12665
Name: count, dtype: int64


In [28]:
import os
print(os.getcwd())

C:\Users\pg293\flight-accident-project\notebook


In [29]:
import pickle

pickle.dump(model, open("../model/model.pkl", "wb"))
pickle.dump(scaler, open("../model/scaler.pkl", "wb"))